# Candle GPT

A small GPT-style decoder-only Transformer for 12-dimensional candle feature vectors.

- Input feature dimension: `12`
- Context length: `10` candles
- Attention heads: `5`
- RoPE positional encoding
- No token lookup embedding layer; the 12D feature vector is projected into model space with a linear layer.
- Output: red/green class logits for the next candle.

Note: standard multi-head attention needs `model_dim % num_heads == 0`, and RoPE needs an even `head_dim`. This notebook uses `model_dim = 60` with `5` heads, giving `head_dim = 12`.


In [17]:
# If needed, install dependencies in the notebook kernel:
# %pip install torch pandas numpy

from pathlib import Path
import bisect
import json
import math
import os
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, random_split

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))
DEVICE


Using device: mps


device(type='mps')

## Variables

In [18]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "historic_candle_feature"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "candle_gpt"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    "upper_wick_to_candle_ratio",
    "lower_wick_to_candle_ratio",
    "body_to_candle_ratio",
    "candle_type_standard",
    "candle_type_doji",
    "candle_type_hammer",
    "candle_type_inverted_hammer",
    "candle_type_spinning_top",
    "candle_type_marubozu",
    "candle_color_green",
    "candle_color_red",
    "log_volume_zscore",
]
FEATURE_MAP = {name: index for index, name in enumerate(FEATURE_COLUMNS)}

CANDLE_COLOR_FEATURES = ["candle_color_green", "candle_color_red"]
CANDLE_COLOR_LABELS = ["green", "red"]
CANDLE_COLOR_INDICES = [FEATURE_MAP[name] for name in CANDLE_COLOR_FEATURES]

INPUT_DIM = 12
OUTPUT_CLASSES = len(CANDLE_COLOR_FEATURES)
CONTEXT_LENGTH = 10
NUM_HEADS = 5
MODEL_DIM = 60
NUM_LAYERS = 3
DROPOUT = 0.15

BATCH_SIZE = 1024 if DEVICE.type == "cuda" else 512
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 30
TRAIN_FRACTION = 0.9
USE_AMP = DEVICE.type == "cuda"

# Keep this True while iterating. Set False for full training.
FAST_DEV_RUN = True
SHUFFLE_FILES = True
MAX_FILES = 12 if FAST_DEV_RUN else None
MAX_ROWS_PER_FILE = 50_000 if FAST_DEV_RUN else None
MAX_TRAIN_BATCHES = 300 if FAST_DEV_RUN else None
MAX_VAL_BATCHES = 80 if FAST_DEV_RUN else None
LOG_EVERY_N_BATCHES = 25

# DataLoader workers help when CPU loading is the bottleneck. If notebooks feel stuck on macOS, set this to 0.
DEFAULT_WORKERS = 0 if DEVICE.type == "mps" else min(4, os.cpu_count() or 0)
NUM_WORKERS = min(2, DEFAULT_WORKERS) if FAST_DEV_RUN else DEFAULT_WORKERS
PIN_MEMORY = DEVICE.type == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0

assert len(FEATURE_COLUMNS) == INPUT_DIM
assert len(CANDLE_COLOR_INDICES) == OUTPUT_CLASSES
assert MODEL_DIM % NUM_HEADS == 0
assert (MODEL_DIM // NUM_HEADS) % 2 == 0

FEATURE_MAP


{'upper_wick_to_candle_ratio': 0,
 'lower_wick_to_candle_ratio': 1,
 'body_to_candle_ratio': 2,
 'candle_type_standard': 3,
 'candle_type_doji': 4,
 'candle_type_hammer': 5,
 'candle_type_inverted_hammer': 6,
 'candle_type_spinning_top': 7,
 'candle_type_marubozu': 8,
 'candle_color_green': 9,
 'candle_color_red': 10,
 'log_volume_zscore': 11}

## Data Loading

Each sample is a 10-candle context. The label is the color of the next candle after that context: `green` or `red`.


In [19]:
class CandleWindowDataset(Dataset):
    def __init__(self, csv_files, feature_columns, context_length, color_indices, max_rows_per_file=None):
        self.feature_columns = feature_columns
        self.context_length = context_length
        self.color_indices = color_indices
        self.arrays = []
        self.window_counts = []
        self.cumulative_counts = []
        self.label_counts = np.zeros(len(color_indices), dtype=np.int64)

        running_total = 0
        skipped = []

        for path in csv_files:
            frame = pd.read_csv(path, nrows=max_rows_per_file)
            missing = [column for column in feature_columns if column not in frame.columns]
            if missing:
                skipped.append((path.name, missing))
                continue

            values = (
                frame[feature_columns]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
                .to_numpy(dtype=np.float32)
            )
            if len(values) <= context_length:
                skipped.append((path.name, ["not enough rows after cleaning"]))
                continue

            next_color_labels = values[context_length:, color_indices].argmax(axis=1)
            self.label_counts += np.bincount(next_color_labels, minlength=len(color_indices))

            self.arrays.append(values)
            count = len(values) - context_length
            self.window_counts.append(count)
            running_total += count
            self.cumulative_counts.append(running_total)

        self.skipped = skipped

    def __len__(self):
        return self.cumulative_counts[-1] if self.cumulative_counts else 0

    def __getitem__(self, index):
        file_index = bisect.bisect_right(self.cumulative_counts, index)
        previous_total = 0 if file_index == 0 else self.cumulative_counts[file_index - 1]
        start = index - previous_total
        values = self.arrays[file_index]

        x = values[start : start + self.context_length]
        next_candle = values[start + self.context_length]
        y = int(next_candle[self.color_indices].argmax())
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


csv_files = sorted(DATA_DIR.glob("*.csv"))
if SHUFFLE_FILES:
    random.Random(42).shuffle(csv_files)
if MAX_FILES is not None:
    csv_files = csv_files[:MAX_FILES]

dataset = CandleWindowDataset(
    csv_files=csv_files,
    feature_columns=FEATURE_COLUMNS,
    context_length=CONTEXT_LENGTH,
    color_indices=CANDLE_COLOR_INDICES,
    max_rows_per_file=MAX_ROWS_PER_FILE,
)

label_distribution = pd.DataFrame(
    {
        "candle_color": CANDLE_COLOR_LABELS,
        "next_candle_count": dataset.label_counts,
    }
)
label_distribution["pct"] = label_distribution["next_candle_count"] / label_distribution["next_candle_count"].sum()
majority_baseline_accuracy = label_distribution["pct"].max()

print(f"CSV files selected: {len(csv_files)}")
print(f"CSV files used: {len(dataset.arrays)}")
print(f"Training windows: {len(dataset):,}")
print(f"Skipped files: {len(dataset.skipped)}")
print(f"Majority-class baseline accuracy: {majority_baseline_accuracy:.3f}")
display(label_distribution)
dataset.skipped[:5]


CSV files selected: 12
CSV files used: 12
Training windows: 457,434
Skipped files: 0
Majority-class baseline accuracy: 0.524


,candle_color,next_candle_count,pct
0,green,217549,0.475586
1,red,239885,0.524414


[]

In [20]:
train_size = int(len(dataset) * TRAIN_FRACTION)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "persistent_workers": PERSISTENT_WORKERS,
}

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

class_weight_values = dataset.label_counts.sum() / (OUTPUT_CLASSES * np.maximum(dataset.label_counts, 1))
CLASS_WEIGHTS = torch.tensor(class_weight_values, dtype=torch.float32, device=DEVICE)

print(f"DataLoader workers: {NUM_WORKERS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches per full epoch: {len(train_loader):,}")
print(f"Val batches per full epoch: {len(val_loader):,}")
print(f"Class weights: {dict(zip(CANDLE_COLOR_LABELS, class_weight_values.round(4)))}")

x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, y_batch.shape)
print(y_batch[:10])


DataLoader workers: 0
Batch size: 512
Train batches per full epoch: 804
Val batches per full epoch: 90
Class weights: {'green': np.float64(1.0513), 'red': np.float64(0.9534)}
torch.Size([512, 10, 12]) torch.Size([512])
tensor([0, 1, 1, 1, 0, 1, 1, 0, 0, 0])


## Model Building

In [21]:
def apply_rope(x):
    # x shape: batch, heads, time, head_dim
    batch, heads, time, head_dim = x.shape
    assert head_dim % 2 == 0

    position = torch.arange(time, device=x.device, dtype=x.dtype)
    inv_freq = 1.0 / (
        10000 ** (torch.arange(0, head_dim, 2, device=x.device, dtype=x.dtype) / head_dim)
    )
    angles = position[:, None] * inv_freq[None, :]
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]

    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rotated = torch.stack(
        [x_even * cos - x_odd * sin, x_even * sin + x_odd * cos],
        dim=-1,
    )
    return rotated.flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, model_dim, num_heads, dropout):
        super().__init__()
        assert model_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        assert self.head_dim % 2 == 0

        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out = nn.Linear(model_dim, model_dim)
        self.dropout_p = dropout

    def forward(self, x):
        batch, time, channels = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(batch, time, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, time, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, time, self.num_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q)
        k = apply_rope(k)

        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=True,
        )
        attended = attended.transpose(1, 2).contiguous().view(batch, time, channels)
        return self.out(attended)


class TransformerBlock(nn.Module):
    def __init__(self, model_dim, num_heads, dropout):
        super().__init__()
        self.attn_norm = nn.LayerNorm(model_dim)
        self.attn = CausalSelfAttention(model_dim, num_heads, dropout)
        self.ffn_norm = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, 4 * model_dim),
            nn.GELU(),
            nn.Linear(4 * model_dim, model_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x


class CandleGPT(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, dropout, output_classes):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, model_dim)
        self.blocks = nn.Sequential(
            *[TransformerBlock(model_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(model_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(model_dim * 2),
            nn.Linear(model_dim * 2, model_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(model_dim, output_classes),
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        last_context_token = x[:, -1, :]
        mean_context_token = x.mean(dim=1)
        pooled = torch.cat([last_context_token, mean_context_token], dim=-1)
        return self.classifier(pooled)

    def predict_proba(self, x):
        return torch.softmax(self(x), dim=-1)


model = CandleGPT(
    input_dim=INPUT_DIM,
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    output_classes=OUTPUT_CLASSES,
).to(DEVICE)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f"Parameters: {parameter_count:,}")


CandleGPT(
  (input_projection): Linear(in_features=12, out_features=60, bias=True)
  (blocks): Sequential(
    (0): TransformerBlock(
      (attn_norm): LayerNorm((60,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=60, out_features=180, bias=True)
        (out): Linear(in_features=60, out_features=60, bias=True)
      )
      (ffn_norm): LayerNorm((60,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): Sequential(
        (0): Linear(in_features=60, out_features=240, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=240, out_features=60, bias=True)
        (3): Dropout(p=0.15, inplace=False)
      )
    )
    (1): TransformerBlock(
      (attn_norm): LayerNorm((60,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=60, out_features=180, bias=True)
        (out): Linear(in_features=60, out_features=60, bias=T

## Training

In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=LEARNING_RATE * 0.1,
)
loss_fn = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


def run_epoch(loader, training, max_batches=None):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_items = 0
    class_correct = torch.zeros(OUTPUT_CLASSES, dtype=torch.long)
    class_total = torch.zeros(OUTPUT_CLASSES, dtype=torch.long)

    for batch_index, (x, y) in enumerate(loader, start=1):
        if max_batches is not None and batch_index > max_batches:
            break

        x = x.to(DEVICE, non_blocking=PIN_MEMORY)
        y = y.to(DEVICE, non_blocking=PIN_MEMORY)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x)
                loss = loss_fn(logits, y)

        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

        batch_items = x.size(0)
        predictions = logits.argmax(dim=-1)
        total_loss += loss.item() * batch_items
        total_correct += (predictions == y).sum().item()
        total_items += batch_items

        for class_index in range(OUTPUT_CLASSES):
            class_mask = y == class_index
            class_total[class_index] += class_mask.sum().detach().cpu()
            class_correct[class_index] += (predictions[class_mask] == class_index).sum().detach().cpu()

        if training and LOG_EVERY_N_BATCHES and batch_index % LOG_EVERY_N_BATCHES == 0:
            print(f"  batch {batch_index:,} | loss {loss.item():.6f}")

    if not total_items:
        return {"loss": float("nan"), "accuracy": float("nan"), "balanced_accuracy": float("nan")}

    per_class_accuracy = class_correct.float() / class_total.clamp_min(1).float()
    balanced_accuracy = per_class_accuracy.mean().item()
    return {
        "loss": total_loss / total_items,
        "accuracy": total_correct / total_items,
        "balanced_accuracy": balanced_accuracy,
        **{f"accuracy_{label}": per_class_accuracy[index].item() for index, label in enumerate(CANDLE_COLOR_LABELS)},
    }


history = []
best_val_loss = float("inf")
best_model_path = ARTIFACT_DIR / "candle_gpt_next_color.pt"

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch: {epoch}")
    train_metrics = run_epoch(train_loader, training=True, max_batches=MAX_TRAIN_BATCHES)
    val_metrics = run_epoch(val_loader, training=False, max_batches=MAX_VAL_BATCHES)
    scheduler.step()

    row = {"epoch": epoch, "learning_rate": scheduler.get_last_lr()[0]}
    row.update({f"train_{name}": value for name, value in train_metrics.items()})
    row.update({f"val_{name}": value for name, value in val_metrics.items()})
    history.append(row)

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        torch.save(model.state_dict(), best_model_path)

    print(
        f"epoch {epoch:02d} | train loss {train_metrics['loss']:.6f} "
        f"| train bal acc {train_metrics['balanced_accuracy']:.3f} "
        f"| val loss {val_metrics['loss']:.6f} "
        f"| val bal acc {val_metrics['balanced_accuracy']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df


Epoch: 1


/var/folders/ss/dk4l3m2j1z3d139kv256wk4r0000gn/T/ipykernel_22548/3414767564.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/var/folders/ss/dk4l3m2j1z3d139kv256wk4r0000gn/T/ipykernel_22548/3414767564.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  batch 25 | loss 0.696172
  batch 50 | loss 0.693979
  batch 75 | loss 0.695777
  batch 100 | loss 0.687583
  batch 125 | loss 0.682947
  batch 150 | loss 0.685603
  batch 175 | loss 0.688712
  batch 200 | loss 0.686021
  batch 225 | loss 0.689883
  batch 250 | loss 0.680693
  batch 275 | loss 0.690311
  batch 300 | loss 0.685824
epoch 01 | train loss 0.688935 | train bal acc 0.534 | val loss 0.691098 | val bal acc 0.514
Epoch: 2
  batch 25 | loss 0.686987
  batch 50 | loss 0.693307
  batch 75 | loss 0.682079
  batch 100 | loss 0.688319
  batch 125 | loss 0.692053
  batch 150 | loss 0.696909
  batch 175 | loss 0.697365
  batch 200 | loss 0.675563
  batch 225 | loss 0.684452
  batch 250 | loss 0.688802
  batch 275 | loss 0.687331
  batch 300 | loss 0.689868
epoch 02 | train loss 0.687831 | train bal acc 0.538 | val loss 0.687492 | val bal acc 0.541
Epoch: 3
  batch 25 | loss 0.684341
  batch 50 | loss 0.690513
  batch 75 | loss 0.682571
  batch 100 | loss 0.690027
  batch 125 | loss 0.

,epoch,learning_rate,train_loss,train_accuracy,train_balanced_accuracy,train_accuracy_green,train_accuracy_red,val_loss,val_accuracy,val_balanced_accuracy,val_accuracy_green,val_accuracy_red
0,1,0.000998,0.688935,0.529766,0.533954,0.620941,0.446968,0.691098,0.535767,0.514122,0.115132,0.913112
1,2,0.000990,0.687831,0.531960,0.537962,0.664046,0.411877,0.687492,0.543140,0.541381,0.508958,0.573804
2,3,0.000978,0.686535,0.538034,0.543339,0.657943,0.428735,0.686285,0.536938,0.543240,0.659404,0.427076
3,4,0.000961,0.686921,0.535007,0.540642,0.654061,0.427223,0.686622,0.538208,0.544657,0.663535,0.425779
4,5,0.000940,0.686627,0.536979,0.542674,0.660139,0.425208,0.686198,0.539819,0.544238,0.625691,0.462785
5,6,0.000914,0.686837,0.535078,0.541057,0.668183,0.413931,0.686515,0.533545,0.542851,0.714389,0.371312
6,7,0.000884,0.686387,0.537539,0.543232,0.664102,0.422361,0.686633,0.542456,0.542701,0.547215,0.538187
7,8,0.000851,0.686570,0.535872,0.542034,0.665621,0.418447,0.686510,0.535547,0.543998,0.699778,0.388217
8,9,0.000815,0.686646,0.534062,0.540415,0.675852,0.404978,0.686463,0.536255,0.544074,0.688213,0.399935
9,10,0.000775,0.686323,0.536563,0.543110,0.682139,0.404081,0.686425,0.541724,0.545840,0.621715,0.469964


## Save Model Artifacts

In [23]:
config = {
    "task": "next_candle_color_classification",
    "feature_columns": FEATURE_COLUMNS,
    "feature_map": FEATURE_MAP,
    "candle_color_features": CANDLE_COLOR_FEATURES,
    "candle_color_labels": CANDLE_COLOR_LABELS,
    "candle_color_indices": CANDLE_COLOR_INDICES,
    "input_dim": INPUT_DIM,
    "output_classes": OUTPUT_CLASSES,
    "context_length": CONTEXT_LENGTH,
    "num_heads": NUM_HEADS,
    "model_dim": MODEL_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "use_amp": USE_AMP,
    "class_weights": CLASS_WEIGHTS.detach().cpu().tolist(),
    "loss": "weighted_cross_entropy",
}

config_path = ARTIFACT_DIR / "config.json"
history_path = ARTIFACT_DIR / "training_history.csv"

config_path.write_text(json.dumps(config, indent=2))
history_df.to_csv(history_path, index=False)

print(best_model_path)
print(config_path)
print(history_path)


/Users/akash/PycharmProjects/AlgoTrade/artifacts/candle_gpt/candle_gpt_next_color.pt
/Users/akash/PycharmProjects/AlgoTrade/artifacts/candle_gpt/config.json
/Users/akash/PycharmProjects/AlgoTrade/artifacts/candle_gpt/training_history.csv


## Test Inference

In [24]:
loaded_model = CandleGPT(
    input_dim=INPUT_DIM,
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    output_classes=OUTPUT_CLASSES,
).to(DEVICE)
loaded_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
loaded_model.eval()

sample_x, sample_y = val_dataset[0]
with torch.no_grad():
    probabilities = loaded_model.predict_proba(sample_x.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)

predicted_class = int(probabilities.argmax().item())
actual_class = int(sample_y.item())

inference_df = pd.DataFrame(
    {
        "candle_color": CANDLE_COLOR_LABELS,
        "predicted_probability": probabilities.numpy(),
        "is_actual_next_color": [index == actual_class for index in range(OUTPUT_CLASSES)],
        "is_predicted_next_color": [index == predicted_class for index in range(OUTPUT_CLASSES)],
    }
)
inference_df


,candle_color,predicted_probability,is_actual_next_color,is_predicted_next_color
0,green,0.559784,False,True
1,red,0.440216,True,False
